#  Sentinel AI Interview Simulator

**An adaptive AI-driven interview intelligence system** that dynamically personalizes technical interviews based on candidate performance, confidence patterns, and domain knowledge.

---
## Phase 1 — Environment Setup & Imports

In [ ]:
# imports

import os
import json
import sqlite3
import base64
import tempfile
from io import BytesIO
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, SVG
import gradio as gr

In [ ]:
# Load environment variables and validate API keys

load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ai_questions_url = os.getenv('AI_QUESTIONS_URL')
ml_questions_url = os.getenv('ML_QUESTIONS_URL')

# Print key prefixes to help with debugging
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if ai_questions_url:
    print(f"AI Questions URL: {ai_questions_url[:50]}...")
else:
    print("AI Questions URL not set")

if ml_questions_url:
    print(f"ML Questions URL: {ml_questions_url[:50]}...")
else:
    print("ML Questions URL not set")

---
## Phase 2 — Multi-Provider API Client

We use the **OpenAI Python library** as a universal client, pointing it at different backends via `base_url`.

| Provider | Model | Audio Support |
|----------|-------|---------------|
| Groq     | Llama 3.3 70B + Whisper STT + gTTS | Pipeline (STT→Brain→TTS) |
| Gemini   | Gemini 2.0 Flash | Native multimodal audio I/O |
| Ollama   | Llama 3.2 Vision 11B | Text-only, no audio |

In [ ]:
# Define API endpoints — one OpenAI library, three backends

groq_url = "https://api.groq.com/openai/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

In [ ]:
# Model registry — each provider has its client, model, and audio capabilities

PROVIDERS = {
    "Groq": {
        "client": OpenAI(api_key=groq_api_key, base_url=groq_url),
        "model": "llama-3.3-70b-versatile",
        "stt_model": "whisper-large-v3",       # Groq-hosted Whisper (ultra-fast)
        "tts": "pipeline",                      # External TTS (gTTS)
        "audio_native": False,                  # No native audio-to-audio
    },
    "Gemini": {
        "client": OpenAI(api_key=google_api_key, base_url=gemini_url),
        "model": "gemini-1.5-flash",
        "stt_model": None,                      # Native — Gemini hears audio directly
        "tts": "native",                        # Native — Gemini generates audio directly
        "audio_native": True,                   # Full multimodal audio I/O
    },
    "Ollama": {
        "client": OpenAI(api_key="ollama", base_url=ollama_url),
        "model": "qwen3.5:9b",
        "stt_model": None,                      # No audio support
        "tts": None,                            # No audio support
        "audio_native": False,                  # TEXT-ONLY — no audio processing
    },
}

print("Providers registered:")
for name, config in PROVIDERS.items():
    audio_status = "🎤 Native Audio" if config["audio_native"] else ("🔊 Pipeline Audio" if config["tts"] else "📝 Text Only")
    print(f"  {name}: {config['model']} — {audio_status}")

In [ ]:
# Helper to get the client and model for a selected provider

def get_client(provider="Groq"):
    """Return (client, model) for the selected provider."""
    p = PROVIDERS[provider]
    return p["client"], p["model"]

# Quick test
client, model = get_client("Groq")
print(f"Default provider: Groq → model={model}")

In [ ]:
# Test: Send a quick message to verify the connection

test_messages = [
    {"role": "user", "content": "Say 'Sentinel AI is online' and nothing else."}
]

response = client.chat.completions.create(model=model, messages=test_messages)
display(Markdown(f"**Provider response:** {response.choices[0].message.content}"))

---
## Phase 3 — SQLite Persistence

Using `db.py` for session and question tracking. Same pattern as Day 4.

In [ ]:
from db import init_db, save_session, save_question, end_session, get_session_summary

# Create the database and tables
init_db()

In [ ]:
# Quick test — create a dummy session and query it

test_sid = save_session(domain="AI", provider="Groq", candidate_name="Test User")
save_question(test_sid, "What is AI?", "Artificial Intelligence", 8, "Good answer", 1)
print(get_session_summary(test_sid))
end_session(test_sid, status="test_completed")

---
## Phase 4 — Domain-Based Interview Question Scraper

The scraper reads URLs from `.env` and fetches questions when the candidate picks AI or ML.
It's triggered as a **tool call** during the interview flow.

In [ ]:
from scraper import scrape_domain_questions

# Test scraping AI questions
result = json.loads(scrape_domain_questions("AI"))
print(f"Domain: {result['domain']} | Source: {result['source'][:50]}... | Questions: {result['count']}")
for i, q in enumerate(result['questions'][:5], 1):
    print(f"  {i}. {q}")

---
## Phase 5 — SVG Question Generator

Pure Python SVG templates — deterministic, instant, free. No LLM needed for generation.

In [ ]:
from svg_gen import generate_svg_question, reveal

# Test each question type
for q_type in ["bar_chart", "shapes", "flowchart"]:
    svg, question, answer = generate_svg_question(q_type)
    print(f"\n{'='*50}")
    print(f"Type: {q_type}")
    print(f"Question: {question}")
    print(f"Answer: {answer}")
    reveal(svg)

---
## Phase 6 — Tool Definitions & Orchestration

5 tools wired up in the OpenAI function-calling format. Same `while tool_calls` pattern as Day 4/5.

In [ ]:
from tools import TOOL_DEFINITIONS, handle_tool_calls, set_session_context

# Show registered tools
print("Registered tools:")
for t in TOOL_DEFINITIONS:
    print(f"  → {t['function']['name']}: {t['function']['description'][:60]}...")

---
## Phase 7 — Chat Memory & Adaptive Interview Engine

**Sliding Window + Summarization Memory** — last 3 exchanges kept verbatim, older ones compressed into a running summary.

In [ ]:
class InterviewMemory:
    """Two-tier memory: recent exchanges + running summary."""
    
    def __init__(self, window_size=3):
        self.window_size = window_size
        self.full_history = []
        self.summary = ""
        self.question_count = 0
        self.scores = []
    
    def add_exchange(self, user_msg, assistant_msg):
        self.full_history.append({"role": "user", "content": user_msg})
        self.full_history.append({"role": "assistant", "content": assistant_msg})
    
    def get_context_messages(self):
        """Return last N exchanges for the API call."""
        return self.full_history[-(self.window_size * 2):]
    
    def get_summary_injection(self):
        """Return summary text to inject into system prompt."""
        if self.summary:
            return f"\n\nInterview context so far: {self.summary}"
        return ""
    
    def update_summary(self, client, model):
        """Ask LLM to summarize the conversation so far."""
        if len(self.full_history) < 4:
            return
        summary_prompt = [{"role": "system", "content": "Summarize this interview conversation in 2-3 sentences. Focus on: topics covered, candidate strengths/weaknesses, and confidence level."},
                          {"role": "user", "content": str(self.full_history)}]
        response = client.chat.completions.create(model=model, messages=summary_prompt)
        self.summary = response.choices[0].message.content
        print(f"Memory updated: {self.summary[:100]}...")

print("InterviewMemory class defined.")

---
## Phase 8 — Streaming (Text + Voice)

Provider-specific audio: Groq=Pipeline, Gemini=Native, Ollama=Text-only.

In [ ]:
# === TEXT STREAMING (all providers) ===

def stream_response(messages, provider="Groq"):
    """Stream a response from the selected provider. Yields text chunks."""
    client, model = get_client(provider)
    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

# === GROQ AUDIO PIPELINE ===

def transcribe_audio_groq(audio_path):
    """Transcribe voice input using Whisper-large-v3 via Groq."""
    groq_client = PROVIDERS["Groq"]["client"]
    with open(audio_path, "rb") as audio_file:
        transcript = groq_client.audio.transcriptions.create(
            model="whisper-large-v3", file=audio_file
        )
    return transcript.text

def speak_response_groq(text):
    """Generate speech using gTTS (free)."""
    from gtts import gTTS
    tts = gTTS(text=text, lang='en')
    fp = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
    tts.save(fp.name)
    return fp.name

# === UNIFIED DISPATCHER ===

def transcribe_audio(audio_path, provider):
    if provider == "Groq":
        return transcribe_audio_groq(audio_path)
    elif provider == "Gemini":
        return transcribe_audio_groq(audio_path)  # Use Groq Whisper as fallback
    return None

def speak_response(text, provider):
    if provider in ["Groq", "Gemini"]:
        return speak_response_groq(text)
    return None

print("Streaming and audio functions defined.")

---
## Phase 9 — System Prompt & Interview Flow

The system prompt controls the interviewer's behavior, termination conditions, and adaptive questioning.

In [ ]:
system_message = """
You are Sentinel, an elite AI interview conductor. Your role is to conduct a focused, adaptive technical interview.

RULES — You MUST follow these strictly:
1. First, welcome the candidate and ask them to choose a domain: AI or ML. Then call the scrape_domain_questions tool.
2. Ask exactly ONE question at a time. Wait for the candidate's response before proceeding.
3. After each answer, evaluate it using the score_answer tool before asking the next question.
4. Adapt difficulty based on performance (use interview context from memory).
5. Stay on-topic. Do NOT deviate from the interview. Politely redirect if needed.
6. Keep scoring private during the interview. However, if the interview is finished AND the candidate explicitly asks for their score, you may call the `get_session_stats` tool to fetch their average score and share it with them.\n
7. Occasionally use the generate_svg_question tool to ask a visual question.

TERMINATION CONDITIONS:
- If the candidate says they don't know (e.g., 'I don't know', 'pass', 'skip'), respond:
  'I understand. Thank you for your honesty. Let's wrap up the interview here.'
  Then call the end_interview tool with reason 'candidate_admitted'.

- If the candidate successfully answers 6 questions, respond:
  'Excellent work! You've demonstrated strong knowledge. Congratulations on completing the interview!'
  Then call the end_interview tool with reason 'completed'.

INTERVIEW STYLE:
- Be professional but warm
- Give brief acknowledgment after each answer
- Mix question types: conceptual, practical, scenario-based, and visual (SVG)
"""

print("System prompt defined.")

In [ ]:
# The main chat callback — integrates everything

# Global state
current_provider = "Groq"
current_session_id = None
memory = InterviewMemory()

def chat(message, history):
    """Main chat callback for Gradio. Handles tool calls in a loop (Day 4/5 pattern)."""
    global current_session_id
    
    client, model = get_client(current_provider)
    
    # Build messages with memory
    sys_msg = system_message + memory.get_summary_injection()
    if current_session_id:
        sys_msg += f"\nCurrent session ID: {current_session_id}"
    
    history_msgs = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": sys_msg}] + history_msgs + [{"role": "user", "content": message}]
    
    response = client.chat.completions.create(model=model, messages=messages, tools=TOOL_DEFINITIONS)
    
    # Handle tool calls in a loop (Day 4/5 pattern)
    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_responses)
        response = client.chat.completions.create(model=model, messages=messages, tools=TOOL_DEFINITIONS)
    
    reply = response.choices[0].message.content
    
    import tools
    if getattr(tools, 'LAST_GENERATED_SVG', None):
        reply = tools.LAST_GENERATED_SVG + "\n\n" + reply
        tools.LAST_GENERATED_SVG = None
    
    
    # Update memory
    memory.add_exchange(message, reply)
    if len(memory.full_history) % 6 == 0:  # Summarize every 3 exchanges
        memory.update_summary(client, model)
    
    return reply

print("Chat function defined.")

---
## Phase 10 — Gradio UI

Full interactive UI using `gr.Blocks` — chat, audio I/O, SVG display, and provider selection.

In [ ]:
# Callbacks for the Gradio UI

def start_interview(provider):
    """Initialize a new interview session."""
    global current_provider, current_session_id, memory
    current_provider = provider
    memory = InterviewMemory()
    current_session_id = save_session(domain="pending", provider=provider)
    set_session_context(current_session_id)
    
    # Get the first message from the interviewer
    client, model = get_client(provider)
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": "Hello, I am ready to begin my interview."}]
    response = client.chat.completions.create(model=model, messages=messages)
    welcome = response.choices[0].message.content
    
    return [{"role": "assistant", "content": welcome}]

def put_message_in_chatbot(message, history):
    """Add user message to chatbot display."""
    return "", history + [{"role": "user", "content": message}]

def process_chat(history):
    """Process the latest user message and get AI response."""
    if not history:
        return history, None
    
    last_user_msg = history[-1]["content"]
    reply = chat(last_user_msg, history[:-1])
    history.append({"role": "assistant", "content": reply})
    
    # Generate audio if supported
    import re
    text_to_speak = re.sub(r'<svg.*?</svg>', '', reply, flags=re.DOTALL).strip()
    audio = speak_response(text_to_speak, current_provider) if PROVIDERS[current_provider]["tts"] else None
    
    return history, audio

def process_voice(audio_path, history):
    """Handle voice input: transcribe → chat → respond."""
    if not audio_path or not PROVIDERS[current_provider].get("stt_model") and not PROVIDERS[current_provider]["audio_native"]:
        return history, None, None
    
    transcript = transcribe_audio(audio_path, current_provider)
    if not transcript:
        return history, None, None
    
    history.append({"role": "user", "content": transcript})
    reply = chat(transcript, history[:-1])
    history.append({"role": "assistant", "content": reply})
    
    import re
    text_to_speak = re.sub(r'<svg.*?</svg>', '', reply, flags=re.DOTALL).strip()
    audio_out = speak_response(text_to_speak, current_provider) if PROVIDERS[current_provider]["tts"] else None
    
    return history, audio_out, None

print("Gradio callbacks defined.")

In [ ]:
# Build the Gradio UI using gr.Blocks (Day 5 pattern)

with gr.Blocks(title="Sentinel AI Interview Simulator", theme=gr.themes.Soft()) as ui:
    gr.Markdown("#  Sentinel AI Interview Simulator")
    gr.Markdown("*An adaptive AI-driven interview intelligence system*")
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=500, label="Interview")
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Settings")
            provider_select = gr.Dropdown(
                ["Groq", "Gemini", "Ollama"], value="Groq", label="AI Provider"
            )
            start_btn = gr.Button("🚀 Start Interview", variant="primary")
    
    with gr.Row():
        audio_input = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Voice Input")
        audio_output = gr.Audio(autoplay=True, label="🔊 Response")
    
    with gr.Row():
        message = gr.Textbox(
            label="Type your answer:",
            placeholder="Or use the microphone above...",
            scale=4
        )
    
    # Event wiring
    start_btn.click(start_interview, inputs=[provider_select], outputs=[chatbot])
    
    message.submit(
        put_message_in_chatbot, [message, chatbot], [message, chatbot]
    ).then(
        process_chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )
    
    audio_input.stop_recording(
        process_voice, inputs=[audio_input, chatbot], outputs=[chatbot, audio_output, audio_input]
    )

ui.launch(inbrowser=True)